# LangChain agent + Geopack MCP — Workflow execution on dataset

Demonstrates the **workflow execution** workflow:

1. User: *"Execute X/Y to Point workflow on dataset 2421"*
2. LLM agent automatically:
   - Gets dataset metadata (columns, geometry, CRS)
   - Searches for and retrieves workflow definition + parameters
   - **Extracts parameter values from dataset metadata**
   - Submits workflow with those parameters
   - Waits for completion
   - Shows results (output dataset/files)

**Key difference from direct SDK usage:** The LLM intelligently maps dataset metadata to workflow parameters — you don't hardcode them.

**Prerequisites:** API running, `notebooks/.env` configured, Phase 1 workflow tools available.

In [8]:
%pip install -q python-dotenv nest_asyncio
%pip install -q -e "../.[mcp,langchain]"

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## ⚠️ IMPORTANT: First Run Instructions

**On first execution of this notebook:**

1. **Run Cell 2 below** (pip install) — installs dependencies
2. **Kernel → Restart Kernel** (important! libraries need to be reloaded)
3. **Run all cells from top in order**

The inprocess MCP dispatcher was recently updated with new workflow tools. Kernel restart ensures they load correctly.


In [2]:
import sys
from pathlib import Path

import nest_asyncio
from dotenv import load_dotenv

nest_asyncio.apply()

NOTEBOOK_DIR = Path.cwd()
SDK_ROOT = NOTEBOOK_DIR.parent if (NOTEBOOK_DIR / "lib").is_dir() else NOTEBOOK_DIR
sys.path.insert(0, str(SDK_ROOT / "src"))
sys.path.insert(0, str(SDK_ROOT / "notebooks" / "lib"))

load_dotenv(SDK_ROOT / "notebooks" / ".env")
load_dotenv(SDK_ROOT / ".env")

from notebook_langchain import setup_notebook_paths, load_env

SDK_ROOT = setup_notebook_paths()
load_env(SDK_ROOT)
print("✅ Setup complete. SDK root:", SDK_ROOT)

✅ Setup complete. SDK root: D:\Works\geopack-geoportal\geopack-geoportal-v2\python-sdk


## Step 1 — Environment & MCP Tools

Load API credentials and discover workflow execution MCP tools.

In [3]:
import os
from notebook_langchain import notebook_mcp_langchain_tools, create_chat_model, WORKFLOW_EXECUTION_SYSTEM_PROMPT

assert os.getenv("GEOPACK_API_URL") and os.getenv("OPENAI_API_KEY"), \
    "⚠️  Set GEOPACK_API_URL and OPENAI_API_KEY in notebooks/.env"

# Cleanup from previous runs if needed
if "mcp_ctx" in globals() and mcp_ctx is not None:
    await mcp_ctx.__aexit__(None, None, None)

# Initialize MCP in-process
print("🔄 Initializing MCP in-process...")
mcp_ctx = notebook_mcp_langchain_tools()
tools, mcp_session, transport = await mcp_ctx.__aenter__()

print(f"✅ Transport: {transport}")
print(f"✅ Discovered {len(tools)} MCP tools:")

# Show all tools with category markers
workflow_tools_found = []
for t in tools:
    if any(kw in t.name.lower() for kw in ["workflow", "submit", "dataset", "wait"]):
        print(f"  - {t.name} ⭐")
        if any(kw in t.name.lower() for kw in ["workflow", "submit"]):
            workflow_tools_found.append(t.name)
    else:
        print(f"  - {t.name}")

# Verify critical tools are present
critical_tools = ["geopack_sdk_get_workflow", "geopack_sdk_submit_workflow", "geopack_sdk_download_workflow_artifact"]
missing = [t for t in critical_tools if t not in [x.name for x in tools]]
if missing:
    print(f"\n⚠️  MISSING TOOLS: {missing}")
    print("   → inprocess_mcp.py may need to be updated")
    print("   → OR kernel restart required (Kernel menu → Restart)")
else:
    print(f"\n✅ All critical workflow tools present!")

🔄 Initializing MCP in-process...
✅ Transport: inprocess
✅ Discovered 14 MCP tools:
  - geopack_sdk_geocode_place
  - geopack_sdk_list_datasets ⭐
  - geopack_sdk_get_dataset ⭐
  - geopack_sdk_query_dataset ⭐
  - geopack_sdk_get_dataset_thumbnail ⭐
  - geopack_sdk_export_dataset ⭐
  - geopack_sdk_get_task
  - geopack_sdk_wait_for_task ⭐
  - geopack_sdk_list_workflows ⭐
  - geopack_sdk_get_workflow ⭐
  - geopack_sdk_submit_workflow ⭐
  - geopack_sdk_get_workflow_run ⭐
  - geopack_sdk_download_workflow_artifact ⭐
  - geopack_sdk_download_generated_file

✅ All critical workflow tools present!


## Step 2 — Agent with workflow execution system prompt

The LLM is instructed to follow the workflow execution sequence:
1. Get dataset metadata
2. Find and get workflow definition
3. Extract parameter values from dataset
4. Submit workflow
5. Wait and show results

In [4]:
from langchain.agents import create_agent

# Try to import the new system prompt; fallback to a simple one if not available
try:
    from notebook_langchain import WORKFLOW_EXECUTION_SYSTEM_PROMPT
    print("✅ WORKFLOW_EXECUTION_SYSTEM_PROMPT loaded from notebook_langchain")
except ImportError as e:
    print(f"⚠️  Could not import WORKFLOW_EXECUTION_SYSTEM_PROMPT: {e}")
    print("   Using fallback system prompt...")
    WORKFLOW_EXECUTION_SYSTEM_PROMPT = """You are a Geoportal workflow assistant. 
When user asks to execute a workflow on a dataset:
1. Get the dataset (geopack_sdk_get_dataset)
2. Find the workflow (geopack_sdk_list_workflows)
3. Get workflow details (geopack_sdk_get_workflow with include_params=true)
4. Submit the workflow (geopack_sdk_submit_workflow)
5. Wait for it (geopack_sdk_wait_for_task)
6. Show results (geopack_sdk_get_workflow_run)
Extract parameter values from dataset metadata automatically."""

llm = create_chat_model()
agent = create_agent(
    llm,
    tools,
    system_prompt=WORKFLOW_EXECUTION_SYSTEM_PROMPT,
)
print("✅ Agent ready with workflow execution system prompt.")

✅ WORKFLOW_EXECUTION_SYSTEM_PROMPT loaded from notebook_langchain
✅ Agent ready with workflow execution system prompt.


In [5]:
# Debug: verify agent has all workflow-related tools
workflow_tools = [t for t in tools if any(kw in t.name.lower() for kw in ["workflow", "submit"])]
print(f"\n🔍 Debug: Agent has {len(workflow_tools)} workflow tools:")
for t in workflow_tools:
    print(f"  ✓ {t.name}")
if not any("get_workflow" in t.name for t in workflow_tools):
    print("\n⚠️  WARNING: geopack_sdk_get_workflow NOT in tools list!")



🔍 Debug: Agent has 5 workflow tools:
  ✓ geopack_sdk_list_workflows
  ✓ geopack_sdk_get_workflow
  ✓ geopack_sdk_submit_workflow
  ✓ geopack_sdk_get_workflow_run
  ✓ geopack_sdk_download_workflow_artifact


## Step 3 — Run workflow request (edit DATASET_ID and WORKFLOW_NAME below)

Edit the user prompt to request workflow execution on your dataset.

In [6]:
# Example: Change these to your dataset and workflow
DATASET_ID = 2294  # xytable
WORKFLOW_NAME = "X/Y to Point"  # Or any other workflow available

USER_PROMPT = f"Execute {WORKFLOW_NAME} workflow on dataset {DATASET_ID}"

print(f"👤 User: {USER_PROMPT}")
print("\n⏳ Running agent (this may take 10-30 seconds)...\n")

agent_result = await agent.ainvoke({"messages": USER_PROMPT})

👤 User: Execute X/Y to Point workflow on dataset 2294

⏳ Running agent (this may take 10-30 seconds)...



HTTP Request: POST https://models.github.ai/inference/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://models.github.ai/inference/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://models.github.ai/inference/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://models.github.ai/inference/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://models.github.ai/inference/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://models.github.ai/inference/chat/completions "HTTP/1.1 413 Payload Too Large"


APIStatusError: Error code: 413 - {'error': {'code': 'tokens_limit_reached', 'message': 'Request body too large for gpt-4.1 model. Max size: 8000 tokens.', 'details': 'Request body too large for gpt-4.1 model. Max size: 8000 tokens.'}}

## Step 4 — Tool chain trace

See what tools the agent called and in what order.

In [6]:
from notebook_langchain import last_assistant_text
from display_datasets import print_tool_trace

messages = agent_result.get("messages", [])
print("Tool chain trace:")
print("=" * 80)
print_tool_trace(messages, max_tools=15)
print("=" * 80)

NameError: name 'agent_result' is not defined

## Step 5 — Assistant reply

Final summary from the agent.

In [ ]:
print("\n🤖 Assistant:\n")
print(last_assistant_text(agent_result))

## Step 6 — Inspect workflow run details (optional)

If you want to see the raw workflow run object with artifacts.

In [ ]:
import json
from langchain_core.messages import ToolMessage

# Find geopack_sdk_get_workflow_run output in tool history
for msg in messages:
    if isinstance(msg, ToolMessage) and msg.tool_name == "geopack_sdk_get_workflow_run":
        try:
            run_result = json.loads(msg.content) if isinstance(msg.content, str) else msg.content
            print("Workflow run details:")
            print(json.dumps(run_result, indent=2)[:2000])  # First 2000 chars
            print("...")
        except:
            pass

In [ ]:
# Cleanup MCP context
await mcp_ctx.__aexit__(None, None, None)
print("✅ MCP context cleaned up.")